# Analyse des estimations d’énergie dans Compar:IA

## Objectif
Ce notebook sert à formuler des questions auxquelles les trois fichiers disponibles (`conversations.parquet`, `votes.parquet` et `reactions.parquet`) peuvent répondre, puis à rendre explicites les limites des données. Les valeurs `kWh` sont des estimations présentes dans le jeu de données : elles ne doivent pas être présentées comme des mesures directes de l’électricité réellement consommée.

Les données sont dans `../compareAI/`. Les trois fichiers partagent `conversation_pair_id`, ce qui permet de relier conversations, votes et réactions.

## Questions déjà retenues

1. **Quels modèles consomment le moins d’énergie pour une tâche comparable ?**
2. **La consommation estimée varie-t-elle selon le type ou la longueur des requêtes ?**

## Trois questions complémentaires proposées

### 3. Le vote des utilisateurs favorise-t-il le modèle dont l’estimation d’énergie est la plus faible ?
**Méthode envisagée :** relier les conversations et les votes par `conversation_pair_id`, écarter les égalités et les votes sans modèle choisi, puis comparer le modèle choisi et l’énergie estimée pour les deux réponses de la même conversation. Présenter la part des votes allant au modèle moins énergivore, avec les effectifs et une analyse des égalités énergétiques.

**Limites :** seuls les échanges ayant reçu un vote sont représentés ; ces votes ne constituent pas un échantillon aléatoire de tous les utilisateurs. Une association entre préférence et énergie ne permet pas de conclure que l’énergie explique la préférence.

### 4. Les réponses mieux évaluées sur certains critères sont-elles associées à une estimation d’énergie plus élevée ?
**Méthode envisagée :** exploiter les indicateurs par modèle dans `votes` (`conv_useful_*`, `conv_complete_*`, `conv_creative_*`, `conv_clear_formatting_*`, ainsi que les indicateurs négatifs), et comparer les estimations d’énergie au sein des conversations pour chaque critère. Rapporter les effectifs par critère et distinguer les valeurs non renseignées des réponses négatives.

**Limites :** les critères sont des appréciations déclarées et ne couvrent pas toutes les dimensions de qualité. Les conversations votées peuvent différer des autres conversations. L’énergie est estimée pour la conversation entière, pas pour un critère isolé.

### 5. Les réactions positives et négatives sont-elles associées à des niveaux différents d’énergie estimée ?
**Méthode envisagée :** relier `reactions` à `conversations` avec `conversation_pair_id`, puis comparer les estimations des réponses associées aux réactions `liked` et `disliked`. Décrire également les critères de réaction disponibles (`useful`, `incorrect`, `superficial`, `complete`, `creative`, `clear_formatting`, etc.) lorsqu’ils sont renseignés.

**Limites :** une conversation peut avoir plusieurs réactions et les réactions ne portent que sur une partie des échanges. L’estimation énergétique disponible est au niveau de la conversation du modèle ; elle ne permet pas d’attribuer précisément une consommation à la réponse ou au tour qui a déclenché la réaction. Les résultats décrivent une association, pas une causalité.

## Inventaire vérifié des fichiers

| Fichier | Lignes | Variables utiles repérées |
|---|---:|---|
| `conversations.parquet` | 410 016 | `conversation_pair_id`, modèles A/B, mode, catégories, langues, nombre de tours, tokens de sortie, estimations `total_conv_a_kwh` et `total_conv_b_kwh` |
| `votes.parquet` | 149 209 | `conversation_pair_id`, modèle choisi, égalité, appréciations par modèle (utile, complet, créatif, format clair, incorrect, superficiel, etc.) |
| `reactions.parquet` | 92 841 | `conversation_pair_id`, modèle/réponse ciblé, tour concerné, aimé/détesté et critères qualitatifs |

Pour les conversations, les deux estimations d’énergie sont disponibles dans 350 270 lignes (85,4 %). Le rapprochement interne donne 144 941 votes associés à une conversation, dont 123 447 avec les deux estimations d’énergie ; parmi ces votes, 98 718 ont un modèle choisi et 46 223 sont indiqués comme égalité. Pour les réactions, 91 111 lignes correspondent à une conversation et 75 658 ont les deux estimations d’énergie.

`conversation_pair_id` est unique dans `conversations` et dans `votes`, mais pas dans `reactions` : il y a plusieurs réactions pour certaines conversations. Les analyses de réactions devront donc préciser si elles portent sur une réaction, un modèle ou une conversation, et éviter de compter les réactions comme des conversations indépendantes.

## Étapes d’analyse à suivre

1. Contrôler la qualité des champs nécessaires et documenter les valeurs manquantes.
2. Définir précisément l’unité d’analyse et les règles de filtrage pour chacune des cinq questions.
3. Comparer les modèles à l’intérieur d’une même conversation quand c’est possible, afin de garder la tâche comparable.
4. Présenter les effectifs avec les statistiques descriptives et les visualisations ; distinguer les résultats observés des limites de couverture.
5. Ne pas conclure à une consommation réelle ou à une relation causale sans mesures ou informations supplémentaires.

## Préparation des analyses

DuckDB lit directement les fichiers Parquet. On compare les estimations dans chaque conversation et on calcule aussi l’énergie estimée pour 1 000 tokens.


In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

candidats = [(Path.cwd() / "../compareAI").resolve(), (Path.cwd() / "compareAI").resolve()]
dossier_donnees = next((p for p in candidats if p.exists()), None)
if dossier_donnees is None:
    raise FileNotFoundError("Dossier compareAI introuvable depuis le répertoire courant")
f_conversations = str(dossier_donnees / "conversations.parquet")
f_votes = str(dossier_donnees / "votes.parquet")
f_reactions = str(dossier_donnees / "reactions.parquet")
fichiers = {"conversations": f_conversations, "votes": f_votes, "reactions": f_reactions}
connexion = duckdb.connect()

print(f"Python {sys.version.split()[0]} | DuckDB {duckdb.__version__} | pandas {pd.__version__}")


Python 3.13.13 | DuckDB 1.5.6 | pandas 3.0.6


**Ce que fait le script :** Initialise les chemins des trois fichiers Parquet, ouvre DuckDB et affiche les versions utilisées.

**Interprétation :** Il s’agit de la configuration du notebook ; cette cellule ne produit pas de résultat analytique.


## Documentation des sources et des données utilisées

Les trois fichiers Parquet sont lus directement depuis `compareAI/` ; aucune source externe n’est jointe. La clé de rapprochement est `conversation_pair_id`. La table `conversations` décrit une paire de réponses et porte les deux estimations d’énergie ; `votes` porte au plus un vote par paire ; `reactions` contient plusieurs événements possibles pour une même paire. Les données imbriquées de dialogue (`conversation_a`/`conversation_b`), les commentaires, les résumés, les mots clés, les paramètres de modèles et les invites système ne sont pas utilisés dans les cinq analyses ci-dessous.

Les champs retenus sont :

| Fichier | Champs retenus | Usage |
|---|---|---|
| `conversations.parquet` | `conversation_pair_id`, `model_a_name`, `model_b_name`, `total_conv_a_kwh`, `total_conv_b_kwh`, tokens A/B | Comparer les deux modèles sur la même demande et calculer kWh/1 000 tokens |
| `conversations.parquet` | `opening_msg`, `categories` | Longueur en mots et type thématique |
| `votes.parquet` | `conversation_pair_id`, `chosen_model_name`, `both_equal`, indicateurs `conv_*_a/b` | Préférence déclarée et critères de qualité par modèle |
| `reactions.parquet` | `conversation_pair_id`, `refers_to_model`, `liked`, `disliked`, critères qualitatifs | Réaction explicite à un modèle ciblé |

Le code suivant permet de consulter le schéma complet réellement lu, y compris les champs non utilisés.


In [2]:
tailles_fichiers = pd.DataFrame([
    {"fichier": nom, "taille_Gio": round(Path(chemin).stat().st_size / 1024**3, 2)}
    for nom, chemin in fichiers.items()
])
tailles_fichiers


,fichier,taille_Gio
0,conversations,2.45
1,votes,0.69
2,reactions,0.90


**Ce que fait le script :** Calcule la taille de chaque fichier source en Gio.

**Interprétation :** Cette sortie décrit le volume à traiter, pas la quantité d’observations ni leur qualité.


In [3]:
from IPython.display import display
fichiers = {
    "conversations": f_conversations,
    "votes": f_votes,
    "reactions": f_reactions,
}
for nom, chemin in fichiers.items():
    print(f"{nom} - {chemin}")
    schema = connexion.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)", [chemin]
    ).df()
    display(schema)


conversations - C:\Users\emeso\Desktop\M2\Projet\compareAI\conversations.parquet


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,timestamp,TIMESTAMP,YES,None,None,None
2,session_hash,VARCHAR,YES,None,None,None
3,visitor_id,VARCHAR,YES,None,None,None
4,mode,VARCHAR,YES,None,None,None
5,custom_models_selection,VARCHAR[],YES,None,None,None
6,conv_turns,BIGINT,YES,None,None,None
7,conversation_pair_id,VARCHAR,YES,None,None,None
8,model_pair_name,VARCHAR,YES,None,None,None
9,opening_msg,VARCHAR,YES,None,None,None


votes - C:\Users\emeso\Desktop\M2\Projet\compareAI\votes.parquet


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,timestamp,TIMESTAMP,YES,None,None,None
2,chosen_model_name,VARCHAR,YES,None,None,None
3,both_equal,BOOLEAN,YES,None,None,None
4,conversation_pair_id,VARCHAR,YES,None,None,None
5,session_hash,VARCHAR,YES,None,None,None
6,conv_comments_a,VARCHAR,YES,None,None,None
7,conv_comments_b,VARCHAR,YES,None,None,None
8,conv_useful_a,BOOLEAN,YES,None,None,None
9,conv_useful_b,BOOLEAN,YES,None,None,None


reactions - C:\Users\emeso\Desktop\M2\Projet\compareAI\reactions.parquet


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,timestamp,TIMESTAMP,YES,None,None,None
2,session_hash,VARCHAR,YES,None,None,None
3,conversation_pair_id,VARCHAR,YES,None,None,None
4,current_conv_turn_when_reacting,BIGINT,YES,None,None,None
5,model_pos,VARCHAR,YES,None,None,None
6,refers_to_model,VARCHAR,YES,None,None,None
7,refers_to_conv_id,VARCHAR,YES,None,None,None
8,system_prompt,VARCHAR,YES,None,None,None
9,response_content,VARCHAR,YES,None,None,None


**Ce que fait le script :** Lit le schéma complet de chaque Parquet avec DuckDB et affiche les noms et types de colonnes.

**Interprétation :** Ces schémas permettent de vérifier la disponibilité des variables avant les analyses. Les scripts suivants n’utilisent qu’un sous-ensemble des colonnes listées dans le dictionnaire des données.


### Étape 1 : Taille, couverture et qualité des clés

Ces contrôles mesurent le nombre de lignes, l’unicité de la clé de paire, les estimations manquantes et la couverture des jointures. Aucun champ manquant n’est imputé. Les lignes sans estimation ou sans tokens positifs sont exclues uniquement des calculs qui en ont besoin.


In [4]:
controle_conversations = connexion.execute(
    """
    SELECT COUNT(*) AS lignes,
      COUNT(DISTINCT conversation_pair_id) AS paires_distinctes,
      COUNT(*) FILTER (WHERE total_conv_a_kwh IS NOT NULL AND total_conv_b_kwh IS NOT NULL) AS deux_energies_disponibles,
      COUNT(*) FILTER (WHERE total_conv_a_kwh IS NULL) AS energie_a_manquante,
      COUNT(*) FILTER (WHERE total_conv_b_kwh IS NULL) AS energie_b_manquante,
      COUNT(*) FILTER (WHERE opening_msg IS NULL OR trim(opening_msg)='') AS demande_manquante,
      COUNT(*) FILTER (WHERE "mode" IS NULL) AS mode_manquant,
      COUNT(*) FILTER (WHERE "mode" IS NOT NULL AND "mode" NOT IN ('random','custom','big-vs-small','reasoning','small-models')) AS mode_atypique,
      MIN(timestamp) AS premiere_date, MAX(timestamp) AS derniere_date
    FROM read_parquet(?)
    """, [f_conversations]
).df()
controle_conversations


,lignes,paires_distinctes,deux_energies_disponibles,energie_a_manquante,energie_b_manquante,demande_manquante,mode_manquant,mode_atypique,premiere_date,derniere_date
0,410016,410016,350270,31699,31616,31,33329,25,2024-10-01 04:18:43.548967,2026-05-18 12:26:35.307219


**Ce que fait le script :** Compte les conversations, vérifie l’unicité de `conversation_pair_id`, mesure les estimations manquantes et repère les valeurs `mode` absentes ou atypiques. Il relève aussi les dates extrêmes.

**Interprétation :** Les 410 016 identifiants de paire sont distincts. Les deux estimations sont disponibles dans 350 270 lignes ; 31 demandes n’ont pas de texte d’ouverture. Le champ `mode` manque dans 33 329 lignes et 25 lignes contiennent une valeur atypique. Les dates observées vont du 1 octobre 2024 au 18 mai 2026.


In [5]:
controle_votes = connexion.execute(
    """
    SELECT COUNT(*) AS lignes_votes,
      COUNT(DISTINCT v.conversation_pair_id) AS paires_distinctes,
      COUNT(*) FILTER (WHERE c.conversation_pair_id IS NOT NULL) AS lignes_raccordees,
      COUNT(*) FILTER (WHERE c.total_conv_a_kwh IS NOT NULL AND c.total_conv_b_kwh IS NOT NULL) AS raccordees_avec_deux_energies,
      COUNT(*) FILTER (WHERE v.chosen_model_name IS NULL) AS sans_modele_choisi,
      COUNT(*) FILTER (WHERE v.both_equal) AS vote_egal
    FROM read_parquet(?) v
    LEFT JOIN read_parquet(?) c USING (conversation_pair_id)
    """, [f_votes, f_conversations]
).df()
controle_votes


,lignes_votes,paires_distinctes,lignes_raccordees,raccordees_avec_deux_energies,sans_modele_choisi,vote_egal
0,149209,149209,144941,123447,47624,47624


**Ce que fait le script :** Compte les votes et les paires distinctes, puis mesure le raccordement à `conversations`, la disponibilité des deux énergies et les votes sans choix ou déclarés égaux.

**Interprétation :** 149 209 votes couvrent 149 209 paires distinctes ; 144 941 se raccordent aux conversations et 123 447 disposent des deux estimations. Les votes égaux et les votes sans modèle choisi doivent être écartés des comparaisons de préférence.


In [6]:
controle_reactions = connexion.execute(
    """
    SELECT COUNT(*) AS lignes_reactions,
      COUNT(DISTINCT r.conversation_pair_id) AS paires_distinctes,
      COUNT(*) FILTER (WHERE c.conversation_pair_id IS NOT NULL) AS lignes_raccordees,
      COUNT(*) FILTER (WHERE r.refers_to_model=c.model_a_name OR r.refers_to_model=c.model_b_name) AS modele_cible_raccorde,
      COUNT(*) FILTER (WHERE r.refers_to_model=c.model_a_name AND c.total_conv_a_kwh IS NOT NULL
                         OR r.refers_to_model=c.model_b_name AND c.total_conv_b_kwh IS NOT NULL) AS estimation_cible_disponible,
      COUNT(*) FILTER (WHERE r.liked) AS aimees,
      COUNT(*) FILTER (WHERE r.disliked) AS detestees
    FROM read_parquet(?) r
    LEFT JOIN read_parquet(?) c USING (conversation_pair_id)
    """, [f_reactions, f_conversations]
).df()
controle_reactions


,lignes_reactions,paires_distinctes,lignes_raccordees,modele_cible_raccorde,estimation_cible_disponible,aimees,detestees
0,92841,52254,91111,91096,83112,63502,29339


**Ce que fait le script :** Compte les réactions et les paires distinctes, puis vérifie le raccordement à une conversation, au modèle ciblé et à son estimation.

**Interprétation :** 92 841 réactions ne représentent que 52 254 paires distinctes, donc plusieurs réactions peuvent porter sur une même conversation. 91 096 réactions ciblent un modèle reconnu et 83 112 ont l’estimation correspondante disponible.


### Étape 2 : Règles de préparation et de comparaison

- **Jointures :** `votes` et `reactions` sont raccordés à `conversations` par `conversation_pair_id`. La clé est unique dans `conversations` et `votes`, mais répétée dans `reactions`.
- **Paires de modèles :** les comparaisons énergétiques utilisent seulement les lignes où les deux noms de modèles et les deux estimations existent.
- **Tokens :** le ratio kWh/1 000 tokens est calculé seulement si le nombre de tokens de sortie est strictement positif. Ce ratio normalise la longueur de la réponse, mais il ne mesure pas une énergie indépendante des estimations du jeu.
- **Longueur :** le texte `opening_msg` est découpé sur les espaces (expression régulière `\s+`), puis classé en 1:10, 11:25, 26:50, 51:200 et plus de 200 mots. Les demandes vides sont exclues.
- **Catégories :** la liste `categories` est déroulée en une ligne par catégorie et par réponse. Une même conversation peut contribuer à plusieurs catégories.
- **Vote :** `both_equal` et l’absence de `chosen_model_name` sont exclus du taux de préférence ; les égalités d’énergie ou noms non rapprochés sont conservés comme catégories de contrôle.
- **Qualité :** pour chaque indicateur, seuls les votes où les modèles ont des états différents (un vrai, un faux) sont comparés. Le delta est l’estimation du modèle portant l’indicateur vrai moins celle de l’autre modèle.
- **Réactions :** seules les valeurs explicitement vraies (`liked = true` ou `disliked = true`) sont comparées. Un `false` n’est pas interprété comme l’autre type de réaction. L’estimation reste à l’échelle de la conversation, pas du message ciblé.
- **Seuils descriptifs :** le classement Q1 exige 1 000 réponses comparables par modèle ; le tableau des catégories exige 1 000 réponses par catégorie. Ces seuils rendent les tableaux plus lisibles et ne constituent pas des tests de significativité.


## Question 1 : Quels modèles ont l’estimation la plus faible pour une tâche comparable ?

On compare les deux réponses dans chaque conversation, puis on agrège la fréquence à laquelle chaque modèle a l’estimation la plus basse. Le tableau retient les modèles avec au moins 1 000 réponses comparables. Le coefficient par 1 000 tokens complète la comparaison de l’énergie totale par demande.


In [7]:
resultats_modeles_par_paire = connexion.execute(
    """
    WITH paires AS (
      SELECT model_a_name ma, model_b_name mb, total_conv_a_kwh ea, total_conv_b_kwh eb,
             total_conv_a_output_tokens ta, total_conv_b_output_tokens tb
      FROM read_parquet(?)
      WHERE model_a_name IS NOT NULL AND model_b_name IS NOT NULL AND model_a_name<>model_b_name
        AND total_conv_a_kwh IS NOT NULL AND total_conv_b_kwh IS NOT NULL
    ), reponses AS (
      SELECT ma modele,ea energie,ta tokens,eb energie_adversaire FROM paires
      UNION ALL SELECT mb,eb,tb,ea FROM paires
    )
    SELECT modele,count(*) nb_reponses_comparees,
      count(*) FILTER(where energie<energie_adversaire) nb_fois_moins_energivore,
      count(*) FILTER(where energie>energie_adversaire) nb_fois_plus_energivore,
      count(*) FILTER(where energie=energie_adversaire) nb_egalites,
      round(100.0*count(*) FILTER(where energie<energie_adversaire)/NULLIF(count(*) FILTER(where energie<>energie_adversaire),0),1) pct_moins_energivore,
      round(median(1000.0*energie/NULLIF(tokens,0)) FILTER(where tokens>0),8) mediane_kwh_1000_tokens
    FROM reponses GROUP BY modele HAVING count(*)>=1000
    ORDER BY pct_moins_energivore DESC,nb_reponses_comparees DESC
    """,[f_conversations]
).df()
resultats_modeles_par_paire.head(15)


,modele,nb_reponses_comparees,nb_fois_moins_energivore,nb_fois_plus_energivore,nb_egalites,pct_moins_energivore,mediane_kwh_1000_tokens
0,chocolatine-2-14b-instruct-v2.0.3-q8,2696,2641,55,0,98.0,0.000088
1,gpt-5.4-nano,2314,2217,83,14,96.4,0.000084
2,lfm2-8b-a1b,4370,4100,169,101,96.0,0.000081
3,lfm2-24b-a2b,1020,940,71,9,93.0,0.000082
4,gpt-4.1-nano,11674,10840,830,4,92.9,0.000107
5,gemma-4-26b-a4b-it,1545,1402,130,13,91.5,0.000084
6,o4-mini,6033,5504,523,6,91.3,0.000101
7,EuroLLM-22B-Instruct-2512,3457,2886,301,270,90.6,0.000106
8,mistral-nemo-2407,10381,9316,1064,1,89.7,0.000094
9,gpt-4o-mini-2024-07-18,12351,10727,1615,9,86.9,0.000101


**Ce que fait le script :** Compare les estimations A et B dans chaque conversation, agrège le nombre de fois où chaque modèle est moins énergivore, et calcule sa médiane de kWh par 1 000 tokens. Le tableau garde les modèles ayant au moins 1 000 réponses comparables.

**Interprétation :** En tête du taux de comparaisons gagnées figurent chocolatine-2-14b-instruct-v2.0.3-q8 (98,0 % sur 2 696), gpt-5.4-nano (96,4 % sur 2 314) et lfm2-8b-a1b (96,0 % sur 4 370). Le classement par énergie totale par tâche diffère du classement par 1 000 tokens : la longueur des réponses et les adversaires rencontrés comptent aussi.


## Question 2 : L’énergie estimée varie-t-elle selon le type ou la longueur de la requête ?

La longueur est le nombre de mots dans `opening_msg`, regroupé en classes. Le type est la catégorie thématique de `categories`. Une conversation peut avoir plusieurs catégories, donc les effectifs ne s’additionnent pas au total.


#### Contrôle de la distribution de longueur

Cette étape chiffre la longue traîne avant de lire les classes de longueur. Elle fournit la médiane, le 95e percentile, le maximum et le nombre de demandes de plus de 200 mots.


In [8]:
distribution_longueur = connexion.execute(
    """
    WITH demandes AS (
      SELECT CASE WHEN opening_msg IS NULL OR trim(opening_msg)='' THEN NULL
        ELSE array_length(regexp_split_to_array(trim(opening_msg),chr(92)||'s+')) END AS mots
      FROM read_parquet(?)
    )
    SELECT COUNT(*) AS demandes_avec_texte,
      MEDIAN(mots) AS mediane_mots,
      QUANTILE_CONT(mots,0.95) AS percentile_95_mots,
      MAX(mots) AS maximum_mots,
      COUNT(*) FILTER (WHERE mots>200) AS demandes_plus_200_mots
    FROM demandes WHERE mots IS NOT NULL
    """, [f_conversations]
).df()
distribution_longueur


,demandes_avec_texte,mediane_mots,percentile_95_mots,maximum_mots,demandes_plus_200_mots
0,409985,16.0,317.0,48385,27678


**Ce que fait le script :** Compte les mots de `opening_msg`, puis calcule la médiane, le 95e percentile, le maximum et le nombre de demandes au-delà de 200 mots.

**Interprétation :** La médiane est de 16 mots, le 95e percentile de 317 mots et le maximum de 48 385 mots. Les demandes très longues forment une longue traîne ; des médianes et des classes sont plus adaptées qu’une moyenne seule.


In [9]:
resultats_longueur = connexion.execute(
    """
    WITH demandes AS (
      SELECT conversation_pair_id,opening_msg,
        CASE WHEN opening_msg IS NULL OR trim(opening_msg)='' THEN NULL
             ELSE array_length(regexp_split_to_array(trim(opening_msg),chr(92)||'s+')) END mots,
        total_conv_a_kwh ea,total_conv_b_kwh eb,total_conv_a_output_tokens ta,total_conv_b_output_tokens tb
      FROM read_parquet(?)
    ), classes AS (
      SELECT *,CASE WHEN mots<=10 THEN '1-10' WHEN mots<=25 THEN '11-25' WHEN mots<=50 THEN '26-50' WHEN mots<=200 THEN '51-200' ELSE '>200' END bande
      FROM demandes WHERE mots IS NOT NULL
    ), reponses AS (
      SELECT conversation_pair_id,mots,bande,ea energie,ta tokens FROM classes
      UNION ALL SELECT conversation_pair_id,mots,bande,eb,tb FROM classes
    )
    SELECT bande,min(mots) minimum_mots,max(mots) maximum_mots,count(*) nb_reponses,
      count(distinct conversation_pair_id) nb_demandes,round(median(energie),8) mediane_kwh,
      round(median(1000.0*energie/NULLIF(tokens,0)),8) mediane_kwh_1000_tokens,
      round(median(tokens),0) mediane_tokens_sortie
    FROM reponses WHERE energie IS NOT NULL AND tokens>0 GROUP BY bande
    ORDER BY CASE bande WHEN '1-10' THEN 1 WHEN '11-25' THEN 2 WHEN '26-50' THEN 3 WHEN '51-200' THEN 4 ELSE 5 END
    """,[f_conversations]
).df()
resultats_longueur


,bande,minimum_mots,maximum_mots,nb_reponses,nb_demandes,mediane_kwh,mediane_kwh_1000_tokens,mediane_tokens_sortie
0,1-10,1,10,247220,132988,0.000260,0.000658,552.0
1,11-25,11,25,258190,139510,0.000387,0.000658,723.0
2,26-50,26,50,102169,55012,0.000509,0.000658,852.0
3,51-200,51,200,92302,49384,0.000722,0.000733,952.0
4,>200,201,48385,50752,27256,0.001174,0.000878,1204.0


**Ce que fait le script :** Répartit les demandes en cinq classes de longueur et calcule, pour les réponses A et B, le nombre de réponses, l’énergie médiane, l’énergie par 1 000 tokens et la médiane des tokens de sortie.

**Interprétation :** L’énergie médiane passe de 0,000260 kWh pour 1:10 mots à 0,001174 kWh au-delà de 200 mots. Les tokens de sortie médians augmentent également de 552 à 1 204. La hausse de l’énergie totale accompagne donc des réponses plus longues ; elle ne démontre pas un effet direct de la longueur d’entrée.


In [10]:
resultats_categories = connexion.execute(
    """
    WITH source AS (
      SELECT conversation_pair_id,categories,total_conv_a_kwh ea,total_conv_b_kwh eb,
             total_conv_a_output_tokens ta,total_conv_b_output_tokens tb FROM read_parquet(?)
    ), reponses AS (
      SELECT conversation_pair_id,categorie,energie,tokens FROM source
      CROSS JOIN UNNEST(categories) u(categorie)
      CROSS JOIN LATERAL (VALUES(ea,ta),(eb,tb)) x(energie,tokens)
    )
    SELECT categorie,count(*) nb_reponses,count(distinct conversation_pair_id) nb_demandes,
      round(median(energie),8) mediane_kwh,
      round(median(1000.0*energie/NULLIF(tokens,0)),8) mediane_kwh_1000_tokens,
      round(median(tokens),0) mediane_tokens_sortie
    FROM reponses WHERE categorie IS NOT NULL AND energie IS NOT NULL AND tokens>0
    GROUP BY categorie HAVING count(*)>=1000 ORDER BY mediane_kwh DESC
    """,[f_conversations]
).df()
resultats_categories


,categorie,nb_reponses,nb_demandes,mediane_kwh,mediane_kwh_1000_tokens,mediane_tokens_sortie
0,History,1423,769,0.000629,0.000682,814.0
1,Law & Justice,54915,29729,0.000510,0.000658,838.0
2,Health & Wellness & Medicine,52695,28451,0.000510,0.000658,849.0
3,Education,237271,127312,0.000502,0.000658,870.0
4,Natural Science & Formal Science & Technology,288196,155096,0.000496,0.000658,806.0
5,Personal Development & Human Resources & Career,76138,41088,0.000480,0.000658,795.0
6,Shopping & Commodity,28768,15509,0.000480,0.000658,691.0
7,Business & Economics & Finance,153757,83683,0.000479,0.000658,822.0
8,Arts,77567,41639,0.000458,0.000658,745.0
9,Entertainment & Travel & Hobby,112404,60349,0.000444,0.000658,710.0


**Ce que fait le script :** Déroule la liste `categories` en observations par catégorie et réponse, puis calcule les mêmes médianes que pour les longueurs. Les catégories avec moins de 1 000 réponses sont exclues du tableau.

**Interprétation :** Les médianes d’énergie totale varient selon la catégorie, par exemple 0,000629 kWh pour History et 0,000328 kWh pour Culture & Cultural geography. La médiane normalisée vaut 0,000658 kWh par 1 000 tokens pour presque toutes les catégories ; les écarts totaux reflètent donc aussi la longueur des réponses et les modèles présents. Les catégories se chevauchent.


## Réponses aux deux premières questions

### Question 1 : Modèles les moins énergivores à tâche comparable

En comparant les deux réponses au sein de chaque conversation et en retenant les modèles vus dans au moins 1 000 réponses comparables, **chocolatine-2-14b-instruct-v2.0.3-q8** a l’estimation la plus faible dans **98,0 %** de ses 2 696 comparaisons. Viennent ensuite **gpt-5.4-nano** (96,4 % sur 2 314), **lfm2-8b-a1b** (96,0 % sur 4 370) et **lfm2-24b-a2b** (93,0 % sur 1 020). Ce classement dépend des adversaires rencontrés par chaque modèle et décrit les paires observées.

En énergie estimée par 1 000 tokens, les médianes les plus faibles parmi les modèles ayant au moins 1 000 comparaisons sont **lfm2-8b-a1b** (0,000081 kWh), **lfm2-24b-a2b** (0,000082), puis **gpt-oss-20b**, **phi-3.5-mini-instruct** et **qwen3-30b-a3b** (environ 0,000083). Le plus faible total par demande n’est donc pas nécessairement le plus faible coefficient par token : la longueur de réponse compte aussi.

### Question 2 : Type et longueur des demandes

**Longueur :** l’énergie médiane estimée augmente avec les classes de mots de la demande : 0,000260 kWh pour 1:10 mots, 0,000387 pour 11:25, 0,000509 pour 26:50, 0,000722 pour 51:200 et 0,001174 au-delà de 200. Les tokens de sortie médians augmentent également (552, 723, 852, 952 puis 1 204). L’énergie par 1 000 tokens reste à 0,000658 kWh jusqu’à 50 mots, puis passe à 0,000733 et 0,000878 dans les classes plus longues. La hausse totale accompagne donc surtout des sorties plus longues ; les différences normalisées restantes peuvent refléter le mélange des modèles.

La longueur des demandes a une longue traîne : médiane **16 mots**, 95e percentile **317 mots**, et 27 678 demandes au-delà de 200 mots. La plus longue en compte 48 385 ; les bornes des classes et les médianes sont donc préférables à une moyenne seule.

**Type :** les catégories les plus fréquentes sont *Natural Science & Formal Science & Technology* (288 196 réponses modèle-catégorie), *Education* (237 271) et *Business & Economics & Finance* (153 757). Les médianes d’énergie totale sont les plus élevées pour *History* (0,000629 kWh, 1 423 réponses), *Law & Justice* et *Health & Wellness & Medicine* (0,000510), et plus faibles pour *Culture & Cultural geography* (0,000328) et *Other* (0,000110). *Other* a aussi une médiane de seulement 131 tokens. Pour presque toutes les catégories, la médiane normalisée est identique (0,000658 kWh/1 000 tokens), ce qui ne montre pas d’effet net de catégorie indépendant du modèle et de la longueur.

**Limites :** les catégories se chevauchent et les modèles sont inégalement répartis entre catégories. Les estimations sont attribuées aux réponses et ne sont pas des mesures directes. On peut décrire des associations, pas attribuer causalement la variation au sujet ou à la longueur.


## Question 3 : Le vote favorise-t-il le modèle à l’estimation énergétique la plus faible ?

On écarte les votes égaux et sans modèle choisi. Les modèles sont comparés à l’intérieur de la même conversation.


In [11]:
resultats_votes = connexion.execute(
    """
    WITH jointure AS (
      SELECT v.chosen_model_name, v.both_equal, c.model_a_name, c.model_b_name,
             c.total_conv_a_kwh ea, c.total_conv_b_kwh eb
      FROM read_parquet(?) v JOIN read_parquet(?) c USING (conversation_pair_id)
      WHERE c.total_conv_a_kwh IS NOT NULL AND c.total_conv_b_kwh IS NOT NULL
    ), classes AS (
      SELECT CASE
        WHEN chosen_model_name=model_a_name AND ea<eb OR chosen_model_name=model_b_name AND eb<ea THEN 'choisi moins énergivore'
        WHEN chosen_model_name=model_a_name AND ea>eb OR chosen_model_name=model_b_name AND eb>ea THEN 'choisi plus énergivore'
        WHEN ea=eb THEN 'estimations égales'
        ELSE 'choix non rapproché'
      END resultat
      FROM jointure WHERE NOT coalesce(both_equal,false) AND chosen_model_name IS NOT NULL
    )
    SELECT resultat, count(*) nb_votes,
           CASE WHEN resultat IN ('choisi moins énergivore','choisi plus énergivore') THEN round(100.0*count(*)/sum(count(*)) FILTER (WHERE resultat IN ('choisi moins énergivore','choisi plus énergivore')) over(),2) END pourcentage_parmi_votes_decisifs
    FROM classes GROUP BY resultat ORDER BY nb_votes DESC
    """, [f_votes, f_conversations]
).df()
resultats_votes


,resultat,nb_votes,pourcentage_parmi_votes_decisifs
0,choisi plus énergivore,47878,57.11
1,choisi moins énergivore,35964,42.89
2,estimations égales,132,NaN
3,choix non rapproché,25,NaN


**Ce que fait le script :** Raccorde les votes aux conversations, écarte les égalités déclarées et les votes sans modèle choisi, puis compare le modèle choisi à celui dont l’estimation d’énergie est la plus faible.

**Interprétation :** Parmi 83 842 votes décisifs, 42,89 % choisissent le modèle à l’estimation la plus faible et 57,11 % celui à l’estimation la plus élevée. Les 132 égalités d’énergie et 25 choix non rapprochés apparaissent séparément. Le résultat est descriptif et porte uniquement sur les conversations votées.


## Question 4 : Les critères de qualité sont-ils associés à une estimation plus élevée ?

Pour chaque critère, on compare les deux modèles dans les conversations où un seul est signalé comme ayant ce critère. Un delta positif signifie que le modèle signalé a une estimation plus élevée.


In [12]:
resultats_qualite = connexion.execute(
    """
    WITH v AS (
      SELECT c.total_conv_a_kwh ea,c.total_conv_b_kwh eb,
        c.total_conv_a_output_tokens ta,c.total_conv_b_output_tokens tb,
        v.conv_useful_a ua,v.conv_useful_b ub,v.conv_complete_a ca,v.conv_complete_b cb,
        v.conv_creative_a cra,v.conv_creative_b crb,
        v.conv_clear_formatting_a cfa,v.conv_clear_formatting_b cfb,
        v.conv_incorrect_a ia,v.conv_incorrect_b ib,
        v.conv_superficial_a sa,v.conv_superficial_b sb,
        v.conv_instructions_not_followed_a na,v.conv_instructions_not_followed_b nb
      FROM read_parquet(?) v JOIN read_parquet(?) c USING (conversation_pair_id)
    ), long AS (
      SELECT 'utile' crit,ua a,ub b,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'complet',ca,cb,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'créatif',cra,crb,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'format clair',cfa,cfb,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'incorrect',ia,ib,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'superficiel',sa,sb,ea,eb,ta,tb FROM v UNION ALL
      SELECT 'consignes non suivies',na,nb,ea,eb,ta,tb FROM v
    ), ecarts AS (
      SELECT crit,
        CASE WHEN a THEN ea-eb ELSE eb-ea END delta_kwh,
        CASE WHEN a THEN 1000.0*ea/NULLIF(ta,0)-1000.0*eb/NULLIF(tb,0)
             ELSE 1000.0*eb/NULLIF(tb,0)-1000.0*ea/NULLIF(ta,0) END delta_kwh_1000
      FROM long WHERE a IS NOT NULL AND b IS NOT NULL AND a<>b
        AND ea IS NOT NULL AND eb IS NOT NULL AND ta>0 AND tb>0
    )
    SELECT crit, count(*) nb_conversations,
      round(100.0*count(*) FILTER(WHERE delta_kwh>0)/count(*),1) pct_signalement_plus_energie,
      round(100.0*count(*) FILTER(WHERE delta_kwh<0)/count(*),1) pct_signalement_moins_energie,
      round(median(delta_kwh),8) delta_median_kwh,
      round(median(delta_kwh_1000),8) delta_median_kwh_1000_tokens
    FROM ecarts GROUP BY crit ORDER BY crit
    """, [f_votes, f_conversations]
).df()
resultats_qualite


,crit,nb_conversations,pct_signalement_plus_energie,pct_signalement_moins_energie,delta_median_kwh,delta_median_kwh_1000_tokens
0,complet,24150,61.2,38.8,0.000110,0.000012
1,consignes non suivies,5164,44.7,55.3,-0.000033,-0.000012
2,créatif,7955,58.2,41.8,0.000076,0.000015
3,format clair,20962,57.0,43.0,0.000052,0.000015
4,incorrect,6230,42.3,57.7,-0.000059,-0.000029
5,superficiel,12935,39.4,60.6,-0.000091,-0.000012
6,utile,19927,55.0,45.0,0.000032,0.000011


**Ce que fait le script :** Pour chaque critère de vote, conserve les conversations où les deux modèles ont des évaluations différentes. Calcule la part où le modèle signalé a plus ou moins d’énergie et le delta médian, brut et par 1 000 tokens.

**Interprétation :** Les critères positifs (utile, complet, créatif, format clair) sont souvent associés à une estimation totale supérieure pour le modèle signalé, tandis que les signaux négatifs (incorrect, superficiel, consignes non suivies) sont plus souvent associés à une estimation inférieure. Les écarts normalisés médians sont faibles ; les résultats ne prouvent pas que la qualité cause une hausse de consommation.


## Question 5 : Les réactions positives et négatives sont-elles associées à des niveaux d’énergie différents ?

On rattache chaque réaction au modèle ciblé et à son estimation dans la conversation. Seules les réactions explicitement aimées ou détestées sont comparées : `false` ne signifie pas automatiquement la réaction opposée.


In [13]:
resultats_reactions = connexion.execute(
    """
    WITH cible AS (
      SELECT r.liked,r.disliked,
        CASE WHEN r.refers_to_model=c.model_a_name THEN c.total_conv_a_kwh
             WHEN r.refers_to_model=c.model_b_name THEN c.total_conv_b_kwh END energie,
        CASE WHEN r.refers_to_model=c.model_a_name THEN c.total_conv_a_output_tokens
             WHEN r.refers_to_model=c.model_b_name THEN c.total_conv_b_output_tokens END tokens
      FROM read_parquet(?) r JOIN read_parquet(?) c USING (conversation_pair_id)
      WHERE r.refers_to_model=c.model_a_name OR r.refers_to_model=c.model_b_name
    ), etiquete AS (
      SELECT 'aimée' reaction,energie,tokens FROM cible WHERE liked=true
      UNION ALL
      SELECT 'détestée',energie,tokens FROM cible WHERE disliked=true
    )
    SELECT reaction,count(*) nb_reactions,
      round(median(energie),8) energie_mediane_kwh,
      round(median(1000.0*energie/NULLIF(tokens,0)),8) energie_mediane_kwh_1000_tokens,
      round(median(tokens),0) mediane_tokens
    FROM etiquete WHERE energie IS NOT NULL AND tokens>0
    GROUP BY reaction ORDER BY reaction
    """, [f_reactions, f_conversations]
).df()
resultats_reactions


,reaction,nb_reactions,energie_mediane_kwh,energie_mediane_kwh_1000_tokens,mediane_tokens
0,aimée,57107,0.000809,0.000857,908.0
1,détestée,25904,0.000286,0.000658,682.0


**Ce que fait le script :** Rattache chaque réaction au modèle visé via `refers_to_model`, puis compare les réactions explicitement `liked = true` et `disliked = true`. Calcule l’énergie médiane brute, par 1 000 tokens et les tokens médians.

**Interprétation :** On observe 57 107 réactions aimées et 25 904 détestées avec énergie et tokens valides. L’énergie médiane par 1 000 tokens est de 0,000857 kWh pour les réponses aimées contre 0,000658 pour les détestées ; celles aimées sont aussi plus longues (908 contre 682 tokens). La différence peut refléter la longueur, le modèle ou le type de réponse, et non la réaction elle-même.


## Résultats descriptifs et limites

### Question 3

Parmi **83 842 votes non égaux** avec les deux estimations, le modèle choisi a l’estimation la plus faible dans **35 964 cas (42,9 %)** ; il a l’estimation la plus élevée dans **47 878 cas (57,1 %)**. Les estimations sont égales dans 132 cas. Le vote ne va donc pas plus souvent au modèle à l’estimation la plus faible dans cet échantillon. Cela ne porte que sur les conversations ayant reçu un vote.

### Question 4

Dans les comparaisons appariées, le modèle signalé comme **utile**, **complet**, **créatif** ou avec un **format clair** a une estimation totale plus élevée dans 55,0 %, 61,2 %, 58,2 % et 57,0 % des cas respectivement. À l’inverse, le modèle signalé comme **incorrect**, **superficiel** ou n’ayant **pas suivi les consignes** a une estimation plus faible dans 57,7 %, 60,6 % et 55,3 % des cas. Les écarts médians par 1 000 tokens restent faibles (environ 0,000011 à 0,000015 kWh pour les critères positifs). Ce sont des associations : elles ne prouvent pas qu’une meilleure qualité exige plus d’énergie.

### Question 5

Après rapprochement au modèle ciblé et filtrage des valeurs d’énergie et de tokens, on obtient **57 107 réactions “aimée”** et **25 904 “détestée”**. L’énergie médiane estimée par 1 000 tokens est **0,000857 kWh** pour les réponses aimées et **0,000658 kWh** pour les réponses détestées. La médiane de longueur est aussi plus élevée pour les réponses aimées (908 contre 682 tokens). L’écart peut donc refléter la longueur, le modèle ou le type de réponse. Une conversation peut avoir plusieurs réactions, et l’énergie est estimée à l’échelle de toute la conversation du modèle, pas du message réagi.

**Conclusion :** ces analyses décrivent des associations dans les votes et réactions observés. Elles ne permettent ni de conclure à un lien causal, ni d’estimer la consommation électrique réelle.
